# Model Context Protocol (MCP) Server Tutorial

This notebook demonstrates how to initialize and run the `MCPServer` in a Jupyter environment.

> [!IMPORTANT]
> **Note:** This notebook is purely for educational demonstration. In practice, the MCP server should always be run as a standalone process via the CLI (such as through [`start_mcp_with_twin.py`](../scripts/start_mcp_with_twin.py) or [`start_mcp_server_cli.py`](../scripts/start_mcp_server_cli.py)). Running it here blocks the notebook and can lead to event-loop conflicts.

## 1. Prerequisites (Database Mode)

To use the MCP server, you must have a Tango Database running and your devices registered and active. If you haven't started them yet, open your terminal and run the following commands:

### Step 1: Start Tango DB
```bash
export TANGO_HOST=localhost:9094
uv run python -m tango.databaseds.database 2
```

### Step 2: Register & Run Devices
You can use the helper script to register default mock devices:
```bash
export TANGO_HOST=localhost:9094
uv run scripts/2_register_devices.py
```
Then start the device servers in separate terminals:
```bash
export TANGO_HOST=localhost:9094
uv run python -m asyncroscopy.hardware.SCAN scan_instance

export TANGO_HOST=localhost:9094
uv run python -m asyncroscopy.detectors.EDS eds_instance

export TANGO_HOST=localhost:9094
uv run python -m asyncroscopy.ThermoMicroscope microscope_instance
```

## 2. Start the MCP Server

Once the database is ready and the devices are running, you can start the MCP Server here.

In [1]:
import sys
import os
from pathlib import Path
import tango

# Resolve project root
notebook_dir = Path.cwd()
project_root = notebook_dir.parent.resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from asyncroscopy.mcp.mcp_server import MCPServer

# Set environment to match the manual startup instructions above
TANGO_PORT = 9094
TANGO_HOST = f"localhost:{TANGO_PORT}"
os.environ["TANGO_HOST"] = TANGO_HOST

# Initialize the MCP Server
print("Initializing MCP Server...")
mcp_server = MCPServer(
    name="TutorialServer",
    tango_host="localhost",
    tango_port=TANGO_PORT
)

print("Discovered Devices:", mcp_server.list_devices())

Initializing MCP Server...
Discovered Devices: ['test/eds/1', 'test/scan/1']


In [3]:
# We have to do this slightly differently because Jupyter owns the event loop
# anyio.run() inside .start() rejects that.
# Call setup() to register tools, then drive run_async() directly.
mcp_server.setup()
await mcp_server.mcp.run_async(transport="streamable-http", host="127.0.0.1", port=8000)

Discovered tools by Tango class:
- EDS: 2
    • State
    • Status
- SCAN: 3
    • Activate
    • State
    • Status


[04/01/26 20:28:04] WARNING  Component already exists: tool:list_devices@                     ]8;id=518309;file:///Users/dominick/asyncroscopy/.venv/lib/python3.14/site-packages/fastmcp/server/providers/local_provider/local_provider.py\local_provider.py]8;;\:]8;id=341734;file:///Users/dominick/asyncroscopy/.venv/lib/python3.14/site-packages/fastmcp/server/providers/local_provider/local_provider.py#192\192]8;;\

Auto-registered tool: list_devices


                    WARNING  Component already exists: tool:EDS_State@                        ]8;id=805766;file:///Users/dominick/asyncroscopy/.venv/lib/python3.14/site-packages/fastmcp/server/providers/local_provider/local_provider.py\local_provider.py]8;;\:]8;id=958284;file:///Users/dominick/asyncroscopy/.venv/lib/python3.14/site-packages/fastmcp/server/providers/local_provider/local_provider.py#192\192]8;;\

                    WARNING  Component already exists: tool:EDS_Status@                       ]8;id=998539;file:///Users/dominick/asyncroscopy/.venv/lib/python3.14/site-packages/fastmcp/server/providers/local_provider/local_provider.py\local_provider.py]8;;\:]8;id=564766;file:///Users/dominick/asyncroscopy/.venv/lib/python3.14/site-packages/fastmcp/server/providers/local_provider/local_provider.py#192\192]8;;\

...

                    WARNING  Component already exists: tool:SCAN_Status@                      ]8;id=344844;file:///Users/dominick/asyncroscopy/.venv/lib/python3.14/site-packages/fastmcp/server/providers/local_provider/local_provider.py\local_provider.py]8;;\:]8;id=540698;file:///Users/dominick/asyncroscopy/.venv/lib/python3.14/site-packages/fastmcp/server/providers/local_provider/local_provider.py#192\192]8;;\


Registered 1 instance method tool(s)
Registered 5 Tango device command tool(s)
Total: 6 tools

All MCP tools available:
  • EDS.State() -> Any
State(DevVoid) -> DevState
-  in (DevVoid): Uninitialised
- out (DevState): Device state
Tango Device Class: EDS
Tango Command: State

  • EDS.Status() -> str
Status(DevVoid) -> DevString
-  in (DevVoid): Uninitialised
- out (DevString): Device status
Tango Device Class: EDS
Tango Command: Status


  • SCAN.Activate(detectors: Annotated[list[str], FieldInfo(annotation=NoneType, required=True, description="List of detectors to activate, e.g. ['haadf', 'bf']. All others are deactivated.")]) -> NoneType
Activate the named detectors and deactivate all others.
Tango Device Class: SCAN
Tango Command: Activate

  • SCAN.State() -> Any
State(DevVoid) -> DevState
-  in (DevVoid): Uninitialised
- out (DevState): Device state
Tango Device Class: SCAN
Tango Command: State

  • SCAN.Status() -> str
Status(DevVoid) -> DevString
-  in (DevVoid): Uninitialised

╭──────────────────────────────────────────────────────────────────────────────╮                  
                 │                                                                              │                  
                 │                                                                              │                  
                 │                         ▄▀▀ ▄▀█ █▀▀ ▀█▀ █▀▄▀█ █▀▀ █▀█                        │                  
                 │                         █▀  █▀█ ▄▄█  █  █ ▀ █ █▄▄ █▀▀                        │                  
                 │                                                                              │                  
                 │                                                                              │                  
                 │                                FastMCP 3.1.1                                 │                  
                 │                            https://gofastmcp.com                             │                  
                 │                                                                              │                  
                 │                    🖥  Server:      TutorialServer, 3.1.1                     │                  
                 │                    🚀 Deploy free: https://fastmcp.cloud                     │                  
                 │                                                                              │                  
                 ╰──────────────────────────────────────────────────────────────────────────────╯                  
                 ╭──────────────────────────────────────────────────────────────────────────────╮                  
                 │                          🎉 Update available: 3.2.0                          │                  
                 │                      Run: pip install --upgrade fastmcp                      │                  
                 ╰──────────────────────────────────────────────────────────────────────────────╯

                    INFO     Starting MCP server 'TutorialServer' with transport 'streamable-http' ]8;id=591571;file:///Users/dominick/asyncroscopy/.venv/lib/python3.14/site-packages/fastmcp/server/mixins/transport.py\transport.py]8;;\:]8;id=291993;file:///Users/dominick/asyncroscopy/.venv/lib/python3.14/site-packages/fastmcp/server/mixins/transport.py#273\273]8;;\
                             on http://127.0.0.1:8000/mcp                                                          

INFO:     Started server process [59746]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)
INFO:     Shutting down
ERROR:    Cancel 0 running task(s), timeout graceful shutdown exceeded
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [59746]
